In [17]:
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [19]:
class CHIMEFRBDataset(Dataset):
    def __init__(self, hdf5_path, catalog_path, target_length):
        self.hdf5_path = hdf5_path
        self.dt = 0.009830400085775182*1000

        self.target_length = target_length
        cat = pd.read_csv(catalog_path, low_memory=False)
        cat["repeater_name"] = cat["repeater_name"].fillna("").str.strip()
        self.repeater_set = set(
            cat.loc[cat["repeater_name"] != "", "tns_name"].str.strip()
        )
        with h5py.File(hdf5_path, "r") as f:
            self.keys = list(f.keys())

        
    @staticmethod
    def _pad_or_crop(wfall, target_length, center_idx):
        n_freq, n_time = wfall.shape
        half = target_length // 2
        start = center_idx - half
        end = start + target_length
        
        if start >= 0 and end <= n_time:
            return wfall[:, start:end]         
        
        src_start = max(start, 0)
        src_end = min(end, n_time)
        out = np.zeros((n_freq, target_length), dtype=wfall.dtype)
        dst_start = src_start - start
        out[:, dst_start:dst_start + (src_end - src_start)] = wfall[:, src_start:src_end]
        return out
    
    
    def __len__(self):
        return len(self.keys)
    
    
    def __getitem__(self, idx):
        key = self.keys[idx]
        
        with h5py.File(self.hdf5_path, "r") as f:
            wfall = f[key]["wfall_plot"][:]
            extent = np.array(f[key]["extent"])
            
        wfall = wfall.astype(np.float32)
        std = wfall.std(axis=1, keepdims=True)
        std[std == 0] = 1.0          # avoid divide-by-zero for masked channels
        wfall = (wfall - wfall.mean(axis=1, keepdims=True)) / std
        

        peak = round(-extent[0] / self.dt)
        wfall = self._pad_or_crop(wfall, self.target_length, peak)
        tensor = torch.from_numpy(wfall)
        label = torch.tensor(int(key in self.repeater_set), dtype=torch.long)
        
        return tensor, label



In [20]:
def make_dataloader(
    hdf5_path: str,
    catalog_path: str,
    target_length: int,
    batch_size: int = 32,
    shuffle: bool = True,
    num_workers: int = 0,
    train_frac: float = 0.8,
    seed: int = 42,
):
    dataset = CHIMEFRBDataset(hdf5_path, catalog_path, target_length)

    n_total = len(dataset)
    n_train = int(n_total * train_frac)
    n_val = n_total - n_train

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [n_train, n_val], generator=generator
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    # Print class balance
    labels = [dataset[i][1].item() for i in range(n_total)]
    n_rep = sum(labels)
    print(f"Dataset: {n_total} bursts | {n_rep} repeaters ({100*n_rep/n_total:.1f}%) "
          f"| {n_total-n_rep} non-repeaters")
    print(f"Train: {n_train} | Val: {n_val}")

    return train_loader, val_loader

In [21]:
TARGET_LENGTH = 128 

train_loader, val_loader = make_dataloader(
    hdf5_path="all_bursts.hdf5",
    catalog_path="chimefrbcat2.csv",
    target_length=TARGET_LENGTH,
    batch_size=32,
    num_workers=0,
)

# Sanity check
for wfall_batch, label_batch in train_loader:
    print(f"Batch shape : {wfall_batch.shape}")
    print(f"Labels      : {label_batch}")
    break


Dataset: 4536 bursts | 981 repeaters (21.6%) | 3555 non-repeaters
Train: 3628 | Val: 908
Batch shape : torch.Size([32, 256, 128])
Labels      : tensor([0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 0])


Need 

- Positional Encoding
- Masking logic
- Encoder blocks
- Decoder blocks
- CLS Token


- Forward
    - Encoder:
        - Embed input frequency snapshot
        - Add positional encoding
        - Mask input
        - Add CLS token
        - Pass through encoder blocks
        - Return x, mask, ids_restore
    - Decoder:
        - Insert masked tokens into the sequence
        - Add positional encoding
        - Pass through decoder blocks
        - Project decoder into prediction space
        - Strip CLS token
        - Return x


In [22]:
class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        
        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0] 
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point. 
        # Edge case e.g.:- 
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan] 
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

In [23]:
class SinusoidalPE(nn.Module):
    def __init__(self, seq_len, embed_dim):
        super().__init__()
        pe = torch.zeros(seq_len, embed_dim)
        pos = torch.arange(seq_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2) * (-np.log(10000) / embed_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe


class FRBMaskedAutoencoder(nn.Module):
    def __init__(self, seq_len, n_freq, embed_dim, contrast_dim=32, mask_ratio=0.25, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.n_freq = n_freq
        self.embed_dim = embed_dim
        self.contrast_dim = contrast_dim
        self.mask_ratio = mask_ratio

        # --- Encoder ---
        self.enc_proj = nn.Linear(n_freq, embed_dim)
        self.enc_drop = nn.Dropout(dropout)
        self.enc_pe = SinusoidalPE(seq_len + 1, embed_dim)
        self.enc_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, nhead=2, dim_feedforward=128,
                                       batch_first=True, dropout=dropout) for _ in range(2)
        ])
        self.enc_norm = nn.LayerNorm(embed_dim)  # Fix 4: re-enable LayerNorm

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dec_pe = SinusoidalPE(seq_len + 1, embed_dim)
        self.dec_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, nhead=2, dim_feedforward=128,
                                       batch_first=True, dropout=dropout) for _ in range(2)
        ])
        self.dec_norm = nn.LayerNorm(embed_dim)  # Fix 4: re-enable LayerNorm
        self.dec_proj = nn.Linear(embed_dim, n_freq)

        self.cls_head = nn.Linear(embed_dim, 1)
        self.cls_drop = nn.Dropout(dropout)
        
        self.proj_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, contrast_dim),
        )

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)

    def mask_input(self, x):
        N, L, D = x.shape
        len_keep = int(L * (1 - self.mask_ratio))

        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1,
                                index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

        mask = torch.ones(N, L, device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore, ids_keep

    def encoder(self, x):
        x = self.enc_drop(self.enc_proj(x))                            # (B, T, E)
        x = x + self.enc_pe.pe[1:self.seq_len + 1] 
        x_vis, mask, ids_restore, ids_keep = self.mask_input(x)

        cls = (self.cls_token + self.enc_pe.pe[0]).expand(x_vis.size(0), -1, -1)
        x_vis = torch.cat([cls, x_vis], dim=1)          # (B, 1+n_keep, E)

        for block in self.enc_blocks:
            x_vis = block(x_vis)
        x_vis = self.enc_norm(x_vis)

        return x_vis, mask, ids_restore

    def decoder(self, x_enc, ids_restore):
        B = x_enc.size(0)
        T = ids_restore.size(1)
        n_keep = x_enc.size(1) - 1                      
        mask_tokens = self.mask_token.expand(B, T - n_keep, -1)

        x_no_cls = x_enc[:, 1:, :]                      # drop CLS
        x_full = torch.cat([x_no_cls, mask_tokens], dim=1)
        x_full = torch.gather(x_full, dim=1,
                              index=ids_restore.unsqueeze(-1).expand(-1, -1, self.embed_dim))

        x_full = x_full + self.dec_pe.pe[1:T + 1]
        cls = x_enc[:, :1, :] + self.dec_pe.pe[0]
        x_full = torch.cat([cls, x_full], dim=1)

        for block in self.dec_blocks:
            x_full = block(x_full)
        x_full = self.dec_norm(x_full)

        recon = self.dec_proj(x_full[:, 1:, :])
        return recon

    def forward(self, x):
        # x: (B, N_FREQ, T)
        x_t = x.permute(0, 2, 1)                       
        x_enc, mask, ids_restore = self.encoder(x_t)
        
        cls_token = x_enc[:, 0, :] # (B, E)
        cls_out = self.cls_drop(self.cls_head(cls_token))
        recon = self.decoder(x_enc, ids_restore) 
        
        proj = nn.functional.normalize(self.proj_head(cls_token), dim=1) # (B, contrast_dim)
        proj = proj.unsqueeze(1) # (B, 1, contrast_dim)
        return recon, cls_out.squeeze(-1), mask, proj


SupConLoss_fn = SupConLoss(temperature=0.07, contrast_mode='all')

def compute_loss(cls_out, x_recon, mask, wfall, labels, proj,
                 alpha=1.0, beta=1.0, gamma=0.1, pos_weight=None, device='cpu'):
    pw = torch.tensor([pos_weight], dtype=torch.float32, device=device) if pos_weight else None
    cls_loss = nn.BCEWithLogitsLoss(pos_weight=pw)(cls_out, labels.float())
    target = wfall.permute(0, 2, 1)                   
    diff = (x_recon - target) ** 2
    recon_loss = (diff * mask.unsqueeze(-1)).sum() / (mask.sum() * wfall.size(1))
    con_loss = SupConLoss_fn(proj, labels=labels, mask=None)
    return alpha * cls_loss + beta * recon_loss + gamma * con_loss, cls_loss, recon_loss, con_loss

In [ ]:
def train_one_epoch(model, loader, optimiser, device):
    model.train()
    total, correct = 0, 0
    running_loss = running_cls = running_recon = running_con = 0.0

    for wfall, labels in loader:
        wfall, labels = wfall.to(device), labels.to(device)

        x_recon, cls_out, mask, proj = model(wfall)
        loss, cls_loss, recon_loss, con_loss = compute_loss(cls_out, x_recon, mask, wfall, labels, proj, device=device)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running_loss  += loss.item()
        running_cls   += cls_loss.item()
        running_recon += recon_loss.item()
        running_con += con_loss.item()
        
        preds = (torch.sigmoid(cls_out.squeeze(-1)) > 0.5).long()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    print(f"  loss={running_loss/n:.4f}  cls={running_cls/n:.4f}  "
          f"recon={running_recon/n:.4f}  con={running_con/n:.4f}  acc={correct/total:.3f}")


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct = 0, 0
    
    # compute batch-wide confusion matrix here
    
    confusion_matrix_global = np.zeros((2, 2), dtype=int)  # [[TN, FP], [FN, TP]]
    
    for wfall, labels in loader:
        wfall, labels = wfall.to(device), labels.to(device)
        x_recon, cls_out, mask, proj = model(wfall)
        preds = (torch.sigmoid(cls_out.squeeze(-1)) > 0.5).long()
        
        
        cf_mat = confusion_matrix(labels.cpu(), preds.cpu(), labels=[0, 1])
        confusion_matrix_global += cf_mat
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        
    print(f"  val acc={correct/total:.3f}")
    print(confusion_matrix_global)

N_EPOCHS = 10
model     = FRBMaskedAutoencoder(
    seq_len=TARGET_LENGTH,
    n_freq=256,
    embed_dim=64,
    mask_ratio=0.25,
).to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=N_EPOCHS)


for epoch in range(N_EPOCHS):
    print(f"Epoch {epoch+1}/{N_EPOCHS}")
    train_one_epoch(model, train_loader, optimiser, device)
    scheduler.step()
    evaluate(model, val_loader, device)

Epoch 1/10
  loss=1.7969  cls=0.5324  recon=0.9205  con=3.4403  acc=0.776
  val acc=0.796
[[700  13]
 [172  23]]
Epoch 2/10
  loss=1.6712  cls=0.4549  recon=0.8752  con=3.4117  acc=0.807
  val acc=0.806
[[656  57]
 [119  76]]
Epoch 3/10
  loss=1.6247  cls=0.4209  recon=0.8638  con=3.3992  acc=0.821
  val acc=0.805
[[677  36]
 [141  54]]
Epoch 4/10
  loss=1.6061  cls=0.4106  recon=0.8565  con=3.3898  acc=0.830
  val acc=0.797
[[630  83]
 [101  94]]
Epoch 5/10
  loss=1.5681  cls=0.3753  recon=0.8554  con=3.3746  acc=0.853
  val acc=0.782
[[623  90]
 [108  87]]
Epoch 6/10
  loss=1.5395  cls=0.3495  recon=0.8549  con=3.3503  acc=0.866
  val acc=0.834
[[697  16]
 [135  60]]
Epoch 7/10
  loss=1.5292  cls=0.3428  recon=0.8526  con=3.3378  acc=0.868
  val acc=0.823
[[648  65]
 [ 96  99]]
Epoch 8/10
  loss=1.5086  cls=0.3239  recon=0.8523  con=3.3227  acc=0.881
  val acc=0.825
[[656  57]
 [102  93]]
Epoch 9/10
  loss=1.5048  cls=0.3224  recon=0.8516  con=3.3081  acc=0.883
  val acc=0.820
[[664 